# ECE 447 - Batch Normalization Reproduction

**Paper**: Ioffe & Szegedy, *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*, ICML 2015

---

## Reporduced Experiments:

### Experiment 1 - MNIST + MLP (Multi-Layered Perceptron) or Fully-Connected (FC) Network + Sigmoid 
#### Part A - Recreation of Experiment/Example from Paper
- **Dataset:**MNIST
- **Model:**3-layer Fullly Connected Network with Sigmoid activations (100 units/neurons in each layer)
- **BN Applied to** each hidden layer's pre-activation (`FC -> BN -> Sigmoid`)
- **Goal**: Show BN trains faster and reaches higher accuracy
- **Measured**: Plot Test Accuracy vs Testing Steps
#### Part B - Validation - Effect of Hidden Width
- **Vary hidden units (nunmber of nuerons)** : [50, 100, 200] 
- **Goal**: Shows BN's benefit holds across network sizes
- **Measured**: Final-accuracy Bar Chart

### Experiment 2 - CIFAR10 CNN, LR Sensitivity (Section 3.2)
#### Part A - Recreation of Experiment/Example from Paper
- **Dataset:**CIFAR-10
- **Model:**CNN - Identical model, Toggeled using `use_bn`
- **Learning rate sweep**: [0.001, 0.01, 0.1]
- **Optimizer**: SGD + momentum (Matches the paper)
- **Goal**: 
    - Without BN -> Diverges at high LR
    - With BN -> Still trains
- **Measured**: Plot Loss vs EPOCHS and Accuracy vs EPOCHS
#### Part B - Validation - Effect of Optimizer
- **Replace SGD+momentum with ADAM at LR=0.1** 
- Adam partially makes the instability due to adaptive LR; The instability introduced by ADAM, reinforeces WHY the paper uses SGD. 

---


### Dependancies
Installing and Importing all dependacies required

In [ ]:
##-----------------------------------------------------------------------##
## DEPENDENCIES
##-----------------------------------------------------------------------##

import sys
!{sys.executable} -m pip install torch torchvision matplotlib --quiet

# Import Dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import random

### CONFIG for Experiment
Global Configs for the reproduction of the paper

In [ ]:
##-----------------------------------------------------------------------##
## CONFIG - GLOBAL VARIABLES 
##-----------------------------------------------------------------------##

# Params for Experiment 1 - Part A: MNIST FC network (reproduces Figure 1)
EXP1_PARAMS = {
    "dataset"      : "MNIST",
    "batch_size"   : 60,       # Paper: "60 examples per mini-batch" (Section 4.1)
    "train_steps"  : 50000,    # Paper: "trained for 50000 steps"    (Section 4.1)
    "hidden_units" : 100,      # Paper: "100 activations each"       (Section 4.1)
    "lr"           : 0.01      # SGD learning rate for MNIST experiment
}

# Params for Experiment 1 - Part B: Hidden width Ablation
EXP1_ABLATION_WIDTHS = [50, 100, 200]

# Params for Experiment 2 - Part A: CIFAR-10 CNN (reproduces Section 3.3 LR sensitivity)
EXP2_PARAMS = {
    "dataset"        : "CIFAR10",
    "learning_rates" : [0.001, 0.01, 0.1],
    "num_epochs"     : 15,
    "batch_size"     : 64,
    "momentum"       : 0.9   # SGD + Momentum matches Paper (Section 4.2)
}

# Params for Experiment 2 - Part B: Optimizer Ablation (SGD vs ADAM at HIgh LR)
EXP2_ABLATION_LR = 0.1
EXP2_ABLATION_OPTS = ["SGD", "ADAM"]



# REPRODUCIBILITY - Fixed SEED
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Data Loading
Function that allows us to load data successfully 

In [43]:
##-----------------------------------------------------------------------##
## LOAD DATA 
##-----------------------------------------------------------------------##

def get_dataset(params):
    dataset = None

    dataset_name = params["dataset"]
    rotation_angle = params.get("rotation_angle", 45)

    base_transformation = [
        torchvision.transforms.v2.ToImage(),
        torchvision.transforms.v2.ToDtype(torch.float32, scale=True)
    ]

    train_subset_size = 50000

    if dataset_name == "MNIST":
        transform = torchvision.transforms.v2.Compose(base_transformation)
        train_ds = torchvision.datasets.MNIST("data", train=True, transform=transform, download=True)
        test_ds  = torchvision.datasets.MNIST("data", train=False, transform=transform, download=True)
        
    elif dataset_name == "RotatedMNIST":
        transform = torchvision.transforms.v2.Compose(base_transformation + [torchvision.transforms.v2.RandomRotation(rotation_angle)])
        train_ds = torchvision.datasets.MNIST("data", train=True, transform=transform, download=True)
        test_ds  = torchvision.datasets.MNIST("data", train=False, transform=transform, download=True)

    elif dataset_name == "CIFAR10":
        transform = torchvision.transforms.v2.Compose(base_transformation)
        train_ds = torchvision.datasets.CIFAR10("data", train=True, transform=transform, download=True)
        test_ds  = torchvision.datasets.CIFAR10("data", train=False, transform=transform, download=True)
        train_subset_size = 40000

    else: 
        raise ValueError(f"Unknown dataset: {dataset_name}")

    train_ds = torch.utils.data.Subset(train_ds, range(train_subset_size))
    return train_ds, test_ds

### Class Definitions
Definig classes for both Experiments: 
- FN_Model
- CNN_Model

In [ ]:
##-----------------------------------------------------------------------##
## EXP 1 - MODEL : FC Network with Sigmoid 
##-----------------------------------------------------------------------##

class FC_Model(nn.Module):
    """
    3-Layer FC Network with sigmoid activation, as described in Section 4.1
    BN is inserted before each sigmoid activation (pre-activation), as per Section 4.1
    bias=False on layers preceding BN: bias is cancelled by BN mean subtraction
    """

    def __init__(self, hidden=100, use_bn=False, num_classes=10):
        super().__init__()
        self.use_bn = use_bn

        # Layer 1: 784 -> 100
        # bias = False when BN follows - BN's beta parameter subsumes the bias
        self.fc1 = nn.Linear(784, hidden, bias=not use_bn)
        self.bn1 = nn.BatchNorm1d(hidden) if use_bn else nn.Identity()

        # Layer 2: 100 -> 100
        self.fc2 = nn.Linear(hidden, hidden, bias=not use_bn)
        self.bn2 = nn.BatchNorm1d(hidden) if use_bn else nn.Identity()

        # Layer 3: 100 -> 100
        self.fc3 = nn.Linear(hidden, hidden, bias=not use_bn)
        self.bn3 = nn.BatchNorm1d(hidden) if use_bn else nn.Identity()

        # Output: 100 -> 10 (NO BN on the output layer)
        self.fc_out = nn.Linear(hidden, num_classes)
    
    def forward(self, x):
        x = x.view(x.size(0), -1) # flatten: (B, 1, 28, 28) -> (B, 784)

        # FC -> BN -> Sigmoid (As Per Paper: z = g(BN(Wu)))
        x = torch.sigmoid(self.bn1(self.fc1(x)))
        x = torch.sigmoid(self.bn2(self.fc2(x)))
        x = torch.sigmoid(self.bn3(self.fc3(x)))


        x = self.fc_out(x)  # Raw logits
        return x

##-----------------------------------------------------------------------##
## EXP 2 - MODEL : CNN for CIFAR-10 
##-----------------------------------------------------------------------##
class CNN_Model(nn.Module):

    """
    CNN for CIFAR-10. The ONLY difference between use_bn=True/False is
    the presence of BatchNorm2d layers — everything else is identical.

    BN placement: Conv -> BN -> ReLU  (pre-activation, per Section 3.2)
    bias=False on Conv layers before BN (BN's beta subsumes the bias).
    """
        
    def __init__(self, use_bn=False, num_classes=10):
        super().__init__()
        self.use_bn = use_bn

        # Block 1: 3 -> 32 channels, 32 x 32 -> 16 x 16
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1, bias=not use_bn)
        self.bn1   = nn.BatchNorm2d(32) if use_bn else nn.Identity() 
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1, bias=not use_bn)
        self.bn2   = nn.BatchNorm2d(32) if use_bn else nn.Identity()
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2: 32->64 Channels, 16 x 16 -> 8 x 8
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1, bias=not use_bn)
        self.bn3   = nn.BatchNorm2d(64) if use_bn else nn.Identity() 
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1, bias=not use_bn)
        self.bn4   = nn.BatchNorm2d(64) if use_bn else nn.Identity()
        self.pool2 = nn.MaxPool2d(2, 2)


        # Block 3: 64 -> 128 Channels,  8 x 8 - > 4 x 4
        self.conv5 = nn.Conv2d(64, 128, 3, padding=1, bias=not use_bn)
        self.bn5   = nn.BatchNorm2d(128) if use_bn else nn.Identity() 
        self.conv6 = nn.Conv2d(128, 128, 3, padding=1, bias=not use_bn)
        self.bn6   = nn.BatchNorm2d(128) if use_bn else nn.Identity()
        self.pool3 = nn.MaxPool2d(2, 2)

        # Classifier (Basically a FC Network)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # CONV -> BN -> ReLU (Pre-actvivation BN, per paper Section 3.2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)

        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.pool3(x)

        x = self.gap(x).view(x.size(0), -1)
        return self.fc(x) # Raw logits


---
## Experiment 1: MNIST FC Network with Sigmoid

The paper's Section 4.1 uses a **3-Layer fully-connected network with sigmoid activations**.
This is critical: Sigmoid saturates badly without BN, which is exactly what the paper demonstrates

**Achitecture per paper**:
```
Input (784) -> FC(100) -> [BN] -> Sigmoid -> FC(100) -> [BN] -> Sigmoid -> FC(100) -> [BN] -> Sigmoid -> FC(10) -> CrossEntropyLoss
```

**BN Placement**: `FC -> BN -> Sigmoid`, this as per Section 3.2: z = g(BN(Wu))

**Note**: bias=False on FC layers before BN - the paper notes the bias is redundant since BN's mean subtraction cancels it.

---
### Part A — Reproduction of Figure 1(a)
Exact paper settings: batch=60, steps=50k, hidden=100, plain SGD.  
**Goal**: show BN trains faster and reaches higher test accuracy.

In [ ]:
##-----------------------------------------------------------------------##
## EXP 1 - TRAINING LOOP
##-----------------------------------------------------------------------##

def run_exp1(use_bn, params, device):
    """
    Trains the FC sigmoid network for 'train_steps' steps.
    Evaluates test accuracy periodically to reproduce Figure 1(a) from the paper.
    Returns: (step_checkpoints, test_accuracies)
    """

    train_ds, test_ds = get_dataset({"dataset":"MNIST"})

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=params["batch_size"], shuffle=True, num_workers=2, pin_memory=True, generator=generator
    )
    test_loader = torch.utils.data.DataLoader(
        test_ds, batch_size=256, shuffle=False, num_workers=2, generator=generator
    )

    model     = FC_Model(hidden=params["hidden_units"], use_bn=use_bn).to(device)
    criterion = nn.CrossEntropyLoss()

    # Paper uses plain SGD for Experiment 1 (NO momentum mentioned in Section 4.1)
    optimizer = torch.optim.SGD(model.parameters(), lr=params["lr"])

    total_steps = params["train_steps"]
    log_interval = total_steps // 20 # ~ 20 checkpoints
    step = 0
    step_log, acc_log = [], []

    def evaluate_test_accuracy():
        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
        
        model.train()
        return correct/total

    # Log Accuracy at step 0 (before any training)
    step_log.append(0)
    acc_log.append(evaluate_test_accuracy())

    model.train()
    while step < total_steps:
        for images, labels in train_loader:
            if step >= total_steps:
                break
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            step += 1

            if step % log_interval == 0:
                acc = evaluate_test_accuracy()
                step_log.append(step)
                acc_log.append(acc)
                print(f"  Step {step:>6d}/{total_steps} | Test Acc: {acc*100:.2f}%")

    return step_log, acc_log

print("\n>  Experiment 1 - Without BN (sigmoid FC network on MNIST)")
steps_no_bn, acc_no_bn = run_exp1(use_bn=False, params=EXP1_PARAMS, device=device)

print("\n>  Experiment 1 - With BN (sigmoid FC network on MNIST)")
steps_bn, acc_bn = run_exp1(use_bn=True,  params=EXP1_PARAMS, device=device)

print("\n  Experiment 1 complete.")

In [ ]:
##-----------------------------------------------------------------------##
## EXP 1 - PLOT (reproduces Figure 1a)
##-----------------------------------------------------------------------##

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(steps_bn, [a * 100 for a in acc_bn], color="#2563EB", lw=2.2, label="With BN", linestyle="-")
ax.plot(steps_no_bn, [a * 100 for a in acc_no_bn], color="#DC2626", lw=2.0, label="Without BN", linestyle="--")

ax.set_xlabel("Training Steps", fontsize=12)
ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_title( "Figure 1(a) Reproduction\n" "MNIST  3-layer FC + Sigmoid  SGD  Test Accuracy vs Steps",fontsize=12, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, EXP1_PARAMS["train_steps"])
ax.set_ylim(0, 100)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K"))

plt.tight_layout()
plt.savefig("exp1_figure1a_reproduction.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp1_figure1a_reproduction.png")

### Part B — Validation - Effect of Hidden Width
The paper uses 100 hidden units. Here we test whether BN's benefit is specific to that choice, or holds across different network sizes.

**Varied Parameter**: hidden units = [50, 100, 200] — everything else is kept identical to Part A.  
**Expectation**: BN outperforms no-BN at all widths; the gap may shrink at larger widths where the no-BN network has more capacity to self-correct.

In [ ]:
##-----------------------------------------------------------------------##
## EXP 1 - PART B : Width Ablation
##-----------------------------------------------------------------------##

exp1b_results = {} # Hold the results: Step and Accuracy for each with 

for width in EXP1_ABLATION_WIDTHS:
    print(f"\n------Width : {width} ------")
    params = {**EXP1_PARAMS, "hidden_units": width}

    print(f"  Without BN")
    b_stp_no, b_acc_no = run_exp1(use_bn=False, params=params, device=device)

    print(f"  With BN")
    b_stp_bn, b_acc_bn = run_exp1(use_bn=True, params=params, device=device)

    exp1b_results[width] = { 
        "no_bn" : (b_stp_no, b_acc_no), 
        "bn"    : (b_stp_bn, b_acc_bn),
    }

    print("\nExperiment 1 Part B complete.")

In [ ]:
##------------------------------------------------------------------------------##
## EXP 1 - PART B : Width Ablation - PLOT (final test accuracy vs hidden width)
##------------------------------------------------------------------------------##

widths = EXP1_ABLATION_WIDTHS
final_no_bn = [exp1b_results[w]["no_bn"][1][-1] * 100 for w in widths]
final_bn = [exp1b_results[w]["bn"][1][-1] * 100 for w in widths]

x = range(len(widths))
bar_width = 0.35

fig, ax = plt.Subplot(figsize=(7,4))
bars_no = ax.bar([i - width/2 for i in x], final_no_bn, bar_width, label="Without BN", color="#DC2626", alpha=0.85)
bars_bn = ax.bar([i + width/2 for i in x], final_no_bn, bar_width, label="With BN", color="#2563EB", alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels([f"{w} units" for w in widths])
ax.set_xlabel("Hidden Layer Width", fontsize=12)
ax.set_ylabel("Final Test Accuracy (%) at 50K steps", fontsize=12)
ax.set_title( "Experiment 1 Part B — BN Benefit Across Hidden Widths\n" "MNIST · 3-layer FC + Sigmoid · SGD", fontsize=12, fontweight="bold")
ax.set_ylim(0, 100)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

# Annotate bar values
for bar in bars_no:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9, color="#DC2626")
for bar in bars_bn:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9, color="#2563EB")

plt.tight_layout()
plt.savefig("exp1b_width_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp1b_width_ablation.png")


---
## Experiment 2: CIFAR-10 CNN — LR Sensitivity (Section 3.3)

This experiment reproduces the paper's central claim from Section 3.3:

**Architecture**: 6-conv-layer CNN with MaxPool and Global Average Pooling. The only difference between the two models is the presence of `BatchNorm2d` layers.  
**BN placement**: `Conv → BN → ReLU`, matching Section 3.2.

> *"Batch Normalization allows us to use much higher learning rates without the risk of divergence."*

**Optimizer**: SGD + momentum=0.9 — deliberately chosen to match the paper and to make the divergence effect visible. Adam's adaptive LR would mask the instability.  
**LR sweep**: [0.001, 0.01, 0.1]  
**Tracked**: training loss, training accuracy, and **test accuracy** each epoch.

**Expected outcome**:
- LR=0.001 → both converge, BN converges faster
- LR=0.01  → both converge, BN more stably  
- LR=0.1   → without BN diverges; with BN still trains

In [ ]:
##-----------------------------------------------------------------------##
## EXP 2 - TRAINING FUNCTION
##-----------------------------------------------------------------------##

def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    One full pass over the training data. Returns (avg_loss, accuracy).
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)

        if not torch.isfinite(loss):
            return float('nan'), 0.0     # diverged — signal early stop

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += outputs.argmax(dim=1).eq(labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, device): 
    """
    Evaluate accuracy on a held-out loader
    """
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += model(images).argmax(dim=1).eq(labels).sum().item()
            total += labels.size(0)

    model.train()
    return correct / total

##-----------------------------------------------------------------------##
## EXP 2 - SWEEP ALL (lr, BN) COMBINATIONS
##-----------------------------------------------------------------------##

generator = torch.Generator()
generator.manual_seed(SEED)

train_ds_c10, test_ds_c10 = get_dataset({"dataset": "CIFAR10"})

cifar_train_loader = torch.utils.data.DataLoader( train_ds_c10, batch_size=EXP2_PARAMS["batch_size"], shuffle=True, num_workers=2, pin_memory=True, generator=generator)
cifar_test_loader = torch.utils.data.DataLoader(test_ds_c10, batch_size=256, shuffle=False, num_workers=2, generator=generator)

# exp2_results[lr][use_bn] = {"train_losses", "train_accs", "test_accs"}
exp2_results = {}

for lr in EXP2_PARAMS["learning_rates"]:
    exp2_results[lr] = {}
    for use_bn in [False, True]:
        tag = f"LR={lr} | BN={'ON' if use_bn else 'OFF'}"
        print(f"\n>  {tag}")

        model     = CNN_Model(use_bn=use_bn).to(device)
        criterion = nn.CrossEntropyLoss()
        # SGD + momentum: matches optimizer style in paper (Section 4.2 uses SGD+momentum)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=EXP2_PARAMS["momentum"])

        train_losses, train_accs, test_accs = [], [], []
        diverged = False

        for epoch in range(EXP2_PARAMS["num_epochs"]):
            tr_loss, tr_acc = train_one_epoch(model, cifar_train_loader, criterion, optimizer, device)
            te_acc = evaluate(model, cifar_test_loader, device) if not diverged else 0.0

            train_losses.append(tr_loss)
            train_accs.append(tr_acc)
            test_accs.append(te_acc)

            loss_str = "NaN" if tr_loss != tr_loss else f"{tr_loss:.4f}"
            print(f"Epoch [{epoch+1:02d}/{EXP2_PARAMS['num_epochs']}]" 
                  f"Train Loss: {loss_str} Train Accuracy: {tr_acc*100:.1f}%"
                  f"Test Accuracy: {te_acc*100:.1f}%")
            
            if tr_loss != tr_loss: # NaN -> Diverged
                remaining = EXP2_PARAMS["num_epochs"] - epoch - 1
                train_losses.extend([float("nan")]*remaining)
                train_accs.extend([0.0] * remaining)
                test_accs.extend([0.0] * remaining)
                print(f"Warning: Diverged at Epoch: {epoch}")
                diverged = True
                break
                
        exp2_results[lr][use_bn] = {
            "train_losses" : train_losses,
            "train_accs" : train_accs,
            "test_accs" : test_accs

        }

print("\n  Experiment 2 complete.")

In [ ]:
##-----------------------------------------------------------------------##
## EXP 2 - PLOT (Section 3.3: LR sensitivity)
##-----------------------------------------------------------------------##

LRS    = EXP2_PARAMS["learning_rates"]
EPOCHS = list(range(1, EXP2_PARAMS["num_epochs"] + 1))
COLOR_BN = "#2563EB"   # blue  — With BN
COLOR_NO = "#DC2626"   # red   — Without BN

fig, axes = plt.subplots(
    nrows=3, ncols=len(LRS),
    figsize=(6 * len(LRS), 12),
    constrained_layout=True
)
fig.suptitle(
    "Experiment 2 Part A — LR Sensitivity: BN vs No BN\n"
    "CIFAR-10  x  CNN  x  SGD + momentum x Solid = With BN  x  Dashed = Without BN",
    fontsize=13, fontweight="bold"
)


for col, lr in enumerate(LRS):
    ax_loss   = axes[0][col]
    ax_train  = axes[1][col]
    ax_test   = axes[2][col]

    for use_bn in [True, False]:
        label = "With BN"    if use_bn else "Without BN"
        color = COLOR_BN     if use_bn else COLOR_NO
        ls    = "-"          if use_bn else "--"
        lw    = 2.2          if use_bn else 1.8
        res   = exp2_results[lr][use_bn]

        losses = exp2_results[lr][use_bn]["losses"]
        accs   = [a * 100 for a in exp2_results[lr][use_bn]["accs"]]

        ax_loss.plot( EPOCHS, res["train_losses"],              color=color, ls=ls, lw=lw, label=label)
        ax_train.plot(EPOCHS, [a*100 for a in res["train_accs"]], color=color, ls=ls, lw=lw, label=label)
        ax_test.plot( EPOCHS, [a*100 for a in res["test_accs"]],  color=color, ls=ls, lw=lw, label=label)

    title_color  = "#DC2626" if lr == 0.1 else "black"
    title_suffix = "  <- diverges without BN" if lr == 0.1 else ""
    ax_loss.set_title(f"LR = {lr}{title_suffix}", fontsize=11, fontweight="bold", color=title_color)


    for ax in [ax_loss, ax_train, ax_test]:
        ax.set_xlabel("Epoch", fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(1, EXP2_PARAMS["num_epochs"])

    ax_loss.set_ylabel("Training Loss",         fontsize=11)
    ax_train.set_ylabel("Training Accuracy (%)", fontsize=11)
    ax_test.set_ylabel("Test Accuracy (%)",      fontsize=11)

    for ax in [ax_train, ax_test]:
        ax.set_ylim(0, 100)

plt.savefig("exp2a_lr_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp2a_lr_sensitivity.png")

---

### Part B - Validation: Effect of Optimizer choice

**Question** : Is the divergence at LR=0.1 specific to SGD, or is it a general property that BN resolves?

**Setup** : Fix LR=0.1 (the problematic case from Part A). Compare SGD+Momentum vs ADAM, both with and without BN. Adam uses adaptive per-parameter learning rates, which natrually dampens large gradient updates - So we expecet ADAM without BN to *not* diverge.

**Why this matters** : If ADAM without BN also converges at LR=0.1, it shows that the instability is optimizer-dependant, not just a property of high LR. This reinforces the paper's choice of SGD to demonstrate BN's effect, and shows that BN's stabilization benifit is most critical in non-adaptive optimizers.

In [ ]:
##-----------------------------------------------------------------------##
## EXP 2 PART B - OPTIMIZER ABLATION (SGD vs Adam at LR=0.1)
##-----------------------------------------------------------------------##

# exp2b_results[optimizer_name][use_bn] = {"train_losses", "train_accs", "test_accs"}
exp2b_results = {}

for opt_name in EXP2_ABLATION_OPTS:
    exp2b_results[opt_name] = {}

    for use_bn in [False, True]:
        tag = f"Optimizer = {opt_name} | BN = {'ON' if use_bn else 'OFF'}"
        print(f"\n>   {tag}")
        
        model = CNN_Model(use_bn=use_bn).to(device)
        criterion = nn.CrossEntropyLoss()

        if opt_name == "SGD":
            optimizer = torch.optim.SGD(model.parameters(), lr=EXP2_ABLATION_LR, momentum=EXP2_PARAMS["momentum"])
        else: # ADMA
            optimizer = torch.optim.Adam(model.parameters(), lr=EXP2_ABLATION_LR)

        train_losses, train_accs, test_accs = [],[],[]
        diverged = False

        for epoch in range(EXP2_PARAMS["num_epochs"]):
            tr_loss, tr_acc = train_one_epoch(model, cifar_train_loader, criterion, optimizer, device)
            te_acc = evaluate(model, cifar_test_loader, device) if not diverged else 0.0

            train_losses.append(tr_loss)
            train_accs.append(tr_acc)
            test_accs.append(te_acc)

            loss_str = "NaN" if tr_loss != tr_loss else f"{tr_loss:.4f}"
            print(f"  Epoch [{epoch+1:02d}/{EXP2_PARAMS['num_epochs']}]  "
                  f"Train Loss: {loss_str}  Train Acc: {tr_acc*100:.1f}%  "
                  f"Test Acc: {te_acc*100:.1f}%")

            if tr_loss != tr_loss:
                remaining = EXP2_PARAMS["num_epochs"] - epoch - 1
                train_losses.extend([float("nan")] * remaining)
                train_accs.extend([0.0] * remaining)
                test_accs.extend([0.0] * remaining)
                print(f"  Warning: Diverged at epoch {epoch+1}.")
                diverged = True
                break

        exp2b_results[opt_name][use_bn] = {
            "train_losses" : train_losses,
            "train_accs"   : train_accs,
            "test_accs"    : test_accs,
        }

print("\nExperiment 2 Part B complete.")

In [ ]:
##-----------------------------------------------------------------------##
## EXP 2 PART B - PLOT (2 optimizers x 2 BN settings)
##-----------------------------------------------------------------------##

EPOCHS   = list(range(1, EXP2_PARAMS["num_epochs"] + 1))
COLOR_BN = "#2563EB"
COLOR_NO = "#DC2626"

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(14, 12),
    constrained_layout=True
)
fig.suptitle(
    f"Experiment 2 Part B — Optimizer Ablation at LR={EXP2_ABLATION_LR}\n"
    "CIFAR-10  x  CNN  x  Solid = With BN  x  Dashed = Without BN",
    fontsize=13, fontweight="bold"
)

for col, opt_name in enumerate(EXP2_ABLATION_OPTS):
    ax_loss  = axes[0][col]
    ax_train = axes[1][col]
    ax_test  = axes[2][col]

    for use_bn in [True, False]:
        label = "With BN"    if use_bn else "Without BN"
        color = COLOR_BN     if use_bn else COLOR_NO
        ls    = "-"          if use_bn else "--"
        lw    = 2.2          if use_bn else 1.8
        res   = exp2b_results[opt_name][use_bn]

        ax_loss.plot( EPOCHS, res["train_losses"],                color=color, ls=ls, lw=lw, label=label)
        ax_train.plot(EPOCHS, [a*100 for a in res["train_accs"]], color=color, ls=ls, lw=lw, label=label)
        ax_test.plot( EPOCHS, [a*100 for a in res["test_accs"]],  color=color, ls=ls, lw=lw, label=label)

    ax_loss.set_title(f"Optimizer: {opt_name}  (LR={EXP2_ABLATION_LR})",
                      fontsize=11, fontweight="bold")

    for ax in [ax_loss, ax_train, ax_test]:
        ax.set_xlabel("Epoch", fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(1, EXP2_PARAMS["num_epochs"])

    ax_loss.set_ylabel("Training Loss",          fontsize=11)
    ax_train.set_ylabel("Training Accuracy (%)", fontsize=11)
    ax_test.set_ylabel("Test Accuracy (%)",      fontsize=11)

    for ax in [ax_train, ax_test]:
        ax.set_ylim(0, 100)

plt.savefig("exp2b_optimizer_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp2b_optimizer_ablation.png")

---
## Summary of Reproduced Findings

### Experiment 1 — MNIST FC Network

**Part A** reproduced Figure 1(a) from the paper.

**Part B** showed that BN's benefit holds across hidden widths of 50, 100, and 200 units.


### Experiment 2 — CIFAR-10 CNN, LR Sensitivity

**Part A** reproduced the central claim of Section 3.3:

**Part B** showed that Adam without BN did *not* diverge at LR=0.


---

### Discrepancies from the Paper

| Aspect | Paper | This Reproduction |
|---|---|---|
|   |   |   | 


---

### Reproducibility Note
All experiments used a fixed random seed (`SEED = 42`) applied to Python, NumPy, and
PyTorch before any data loading or model initialization. Results may vary slightly across
hardware due to non-deterministic CUDA operations.